## 4.1 RNN网络结构 - RNN层

#### 1. 这一节我们要解决什么问题 🎯

在开始看公式之前，这一节应该先回答一个更基础的问题：  
RNN 层到底是什么？

也就是说，我们先不要一上来就讨论输出结果，而是先从整个模型结构角度理解：

* RNN 在神经网络里属于哪一层？
* 它和我们之前学过的全连接层、卷积层是什么关系？
* 它到底负责做什么？
* 为什么说它是“处理序列的特征提取层”？

这一节学完后，应该先建立这样一个清晰认知：

> RNN层本质上也是神经网络中的一层，只不过它不是提取空间特征，而是提取时序特征。


#### 2. 先从整体上理解：RNN层在整个模型中的位置 🧠

##### 2.1 RNN层本质上也是“网络中的一层”

我们以前学过：

* 在 MLP 中，有全连接层（Linear）
* 在 CNN 中，有卷积层（Conv）
* 而在处理序列数据时，对应的核心层就是：RNN层（Recurrent Neural Network Layer）

所以从网络结构上说，RNN层并不是什么脱离神经网络体系的新东西，它仍然只是：

> 神经网络中的一种层。

只是它处理的数据类型更特殊，它不是为普通表格数据设计的，也不是为图像的空间结构设计的，而是专门为：

* 文本序列
* 时间序列
* 语音序列
* 任意按时间或顺序排列的数据

##### 2.2 RNN层和我们以前学过的层有什么不同？

最核心的区别在于：

* 全连接层：主要处理静态特征
* 卷积层：主要提取空间局部特征
* RNN层：主要提取时序依赖特征

也就是说：

> RNN层关注的不是“当前这个输入本身”，而是“当前输入和前面输入之间的关系”。

比如一句话：

$I\ love\ deep\ learning$

对于普通网络来说，如果把每个词独立看，它不知道单词之间的先后关系。  
但对于 RNN 来说，它会按顺序一个一个处理：

* 先读 $I$
* 再读 $love$
* 再读 $deep$
* 再读 $learning$

所以它能保留“前文信息”，这就是 RNN层最核心的价值。

##### 2.3 RNN层在模型里的作用是什么？

RNN层最本质的作用可以概括为一句话：

> 对序列数据进行时序特征提取。

就像 CNN 的卷积层负责从图像中提取边缘、纹理、形状等空间特征一样，  
RNN层负责从序列中提取：

* 顺序关系
* 上下文关系
* 历史依赖
* 时间动态变化规律

所以如果从整个模型角度看，一个典型的 RNN 模型结构往往是：

输入序列 $\rightarrow$ RNN层提取时序特征 $\rightarrow$ 全连接层输出结果

也就是说：

> RNN层本身通常不是最终输出层，而是一个“时序特征提取层”。

#### 3. RNN层的模型结构本质上是什么 🔁

##### 3.1 先不要把 RNN层想得太复杂

RNN 最核心的结构，其实可以压缩成一句话：

> RNN层是一个会在时间上重复使用的神经网络计算层。

这里有两个关键词一定要抓住：

* 重复使用
* 时间传递

所谓重复使用，就是同一个 RNN 计算单元，会在序列的每一个时间步反复执行。  
所谓时间传递，就是当前这一步的计算结果，会传递给下一步继续使用。

所以 RNN层和普通层最大的不同是：

> 它不是只算一次，而是会沿着序列一步一步反复调用计算单元计算。

##### 3.2 RNN层中的“小方块”其实不是一个神经元

很多人在刚开始学 RNN 时，看结构图会误以为：

> “图上这个小方块是不是就是一个神经元？”

其实不是。

图中的一个 RNN 小方块，表示的不是一个单独神经元，而是：

> 一个完整的 RNN 计算单元，也可以理解为一个 RNN cell。

它内部其实完成的是一整套神经网络计算：

* 接收输入
* 与上一时刻状态结合
* 做线性变换
* 做非线性激活
* 生成新的隐藏状态
* 必要时再生成输出

所以这个小方块本质上不是“一个点”，而是“一个层在单个时间步上的一次计算展开”。

##### 3.3 为什么说它本质上仍然是神经网络计算？

因为它内部完全具备神经网络的基本要素：

* 第一，它有输入：
  * 当前输入 $x_t$
  * 上一时刻状态 $h_{t-1}$
* 第二，它有参数：
  * $W_x$
  * $W_h$
  * $b$
* 第三，它有线性变换：
  * $W_x x_t + W_h h_{t-1} + b$
* 第四，它有非线性激活：
  * $tanh$
  * $ReLU$
  * $sigmoid$
  * $softmax$
* 第五，它可以通过损失函数和反向传播训练。

所以从本质上说：

> RNN层并不是“像神经网络”，它本身就是神经网络，只不过它是一种专门面向序列的神经网络层。


#### 4. RNN层最核心的内容：隐藏状态的递推

##### 4.1 这一节为什么要突出隐藏状态？

因为 RNN层本体最核心的任务，不是直接产生最终任务输出，而是：

> 根据当前输入和上一时刻状态，计算当前时刻的隐藏状态。

也就是说，在 RNN层这一节里，我们最应该关注的是：

* $x_t$ 是什么
* $h_{t-1}$ 是什么
* $h_t$ 是怎么计算出来的
* $h_t$ 为什么能表示“到当前为止的信息”

至于最终输出结果，例如分类结果、预测值，这些通常是：

> 隐藏状态再经过后续全连接层映射得到的。

##### 4.2 RNN层的核心产物是什么？

RNN层在每一个时间步最核心的产物就是：

> 当前隐藏状态 $h_t$

它可以理解为：

> 模型在第 $t$ 个时间步，对“到目前为止整个历史信息”的内部表示。

所以这一层最关键的是：

* $h_1$
* $h_2$
* $h_3$
* $\dots$
* $h_t$

这些隐藏状态沿着时间不断传递，构成了 RNN 的“记忆链”。

##### 4.3 最核心的单步计算公式

RNN层最经典的一条公式是：

$h_t = f(W_x x_t + W_h h_{t-1} + b)$

如果激活函数使用 $tanh$，那么也可以写成：

$h_t = tanh(W_x x_t + W_h h_{t-1} + b)$

这条公式就是这一小节最核心的公式。  
它表示：

> 当前隐藏状态 $h_t$，是由当前输入 $x_t$ 和上一时刻隐藏状态 $h_{t-1}$ 共同决定的。

#### 5. 输入 $x_t$ 和隐藏状态 $h_t$ 分别是什么？

##### 5.1 输入 $x_t$

$x_t$ 表示第 $t$ 个时间步的输入。

这里的输入不一定是一个数字，更常见的是：

* 一个向量
* 一个 embedding 向量
* 一个特征向量

例如一句话：

$I\ love\ deep\ learning$

如果按单词一个一个送入 RNN，那么：

* $x_1 = I$
* $x_2 = love$
* $x_3 = deep$
* $x_4 = learning$

更准确地说，在实际模型中，这些词通常先会变成向量，再送入 RNN。

所以：

> $x_t$ 通常表示第 $t$ 个时间步的输入特征向量。

##### 5.2 上一时刻隐藏状态 $h_{t-1}$

$h_{t-1}$ 表示前一个时间步保留下来的隐藏状态。

它不是原始输入，而是模型在前一时刻已经计算出来的“内部记忆”。

你可以把它理解为：

> 前面所有历史信息，到上一时刻为止的压缩总结。

##### 5.3 当前隐藏状态 $h_t$

$h_t$ 表示当前时间步的隐藏状态。

它是 RNN层在当前时刻最重要的输出，也是下一时刻继续使用的输入之一。

它可以理解为：

> 模型在当前时刻保留下来的内部记忆。

这个记忆不是原始输入本身，而是对“到当前为止的信息”的一种压缩表示。

所以隐藏状态有两个核心作用：

* 保存到当前为止的上下文信息
* 把信息传递给下一个时间步

所以 RNN 之所以能“记住前面”，本质上就是依赖：

> 隐藏状态在时间上的传递。


#### 6. 如何理解隐藏状态公式中的每一部分？

对于公式：

$h_t = tanh(W_x x_t + W_h h_{t-1} + b)$

我们可以拆开理解。

##### 6.1 $W_x x_t$：当前输入带来的信息

这一部分表示：

> 当前输入 $x_t$ 对当前隐藏状态的影响。

也就是说，RNN 在当前时刻一定会看当前输入本身。

##### 6.2 $W_h h_{t-1}$：历史信息带来的影响

这一部分表示：

> 上一时刻隐藏状态 $h_{t-1}$ 对当前隐藏状态的影响。

这正是 RNN 和普通前馈网络最本质的区别。

因为普通网络通常只看当前输入，而 RNN 还会额外接收一份“历史记忆”。

##### 6.3 $b$：偏置项

这一部分和普通神经网络一样，是偏置项，用来辅助模型更灵活地拟合数据。

##### 6.4 $tanh$：非线性激活

前面的线性组合：

$W_x x_t + W_h h_{t-1} + b$

只是线性结果。

还需要经过激活函数，才能得到当前隐藏状态：

$h_t = tanh(\dots)$

这样模型才能具备非线性表达能力。

### 7. RNN层的多时间步递推过程

##### 7.1 第一个时间步的 $h_0$ 从哪来？

通常有两种方式：

* 初始化为全 $0$ 向量
* 初始化为可学习参数

在基础学习阶段，我们通常先理解成：

$h_0 = 0$

也就是一开始没有历史记忆。

##### 7.2 状态如何在时间上传递？

例如一个长度为 $4$ 的序列：

* 第一步：$h_1 = tanh(W_x x_1 + W_h h_0 + b)$
* 第二步：$h_2 = tanh(W_x x_2 + W_h h_1 + b)$
* 第三步：$h_3 = tanh(W_x x_3 + W_h h_2 + b)$
* 第四步：$h_4 = tanh(W_x x_4 + W_h h_3 + b)$

于是就形成了一条沿时间传播的状态链。

所以 hidden state 可以理解成：

> 信息在时间维度上的接力棒。

##### 7.3 为什么 RNN层能处理序列？

因为它不是孤立地看每个输入，而是在处理当前输入 $x_t$ 时，同时结合：

* 当前输入本身
* 前面累计下来的历史信息

所以当前时刻的隐藏状态，不只是“当前信息”，而是：

> 到当前为止的历史总结。

这就是 RNN 具备序列建模能力的根本原因。

#### 8. RNN层中的参数共享

##### 8.1 什么叫参数共享？

虽然序列有很多时间步，例如：

* $x_1$
* $x_2$
* $x_3$
* $x_4$

但 RNN 在每一个时间步中使用的权重矩阵通常是同一组：

* 同一个 $W_x$
* 同一个 $W_h$
* 同一个 $b$

也就是说：

> 不同时间步，计算规则一样；只是输入和状态不同。

这就叫做参数共享。

##### 8.2 为什么要共享参数？

因为序列问题中，我们希望模型在每一个位置都使用同样的处理逻辑。

比如读句子时：

* 读第 $1$ 个词
* 读第 $2$ 个词
* 读第 $10$ 个词

虽然位置不同，但“处理一个时间步输入”的方式应该是一致的。

参数共享的好处有：

* 参数量更少
* 更适合处理变长序列
* 强调“同一种规则反复应用”


#### 9. 本节总结 🧠

这一节最重要的，不是先讲输出结果，而是先建立这个核心认知：

> RNN层本质上是一个时序特征提取层，它最核心的工作是递推计算隐藏状态。

也就是说，RNN层真正关注的是：

$x_t + h_{t-1} \rightarrow h_t$

而不是直接关注最终任务输出。

最终输出结果通常是：

> 隐藏状态再送入后续全连接层后得到的。